# 02 — Feature Engineering (Early Window)

Build **weeks 1–3** features (`date <= 21`) for early at-risk detection.

**Key findings from 01:**
- 32,593 enrollments; **52.8% at-risk** (Withdrawn + Fail)
- Strongest full-term signals: `active_days` (−0.63), `total_clicks` (−0.48), `mean_score` (−0.43)
- 20.8% missing `mean_score` (mostly withdrawn students with no submissions)
- `late_submission_count` is **lower** for at-risk students — withdrawn learners stop submitting, so lateness is a weak/noisy signal

This notebook restricts features to the **first 21 days** to avoid leakage from post-withdrawal behavior.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = PROJECT_ROOT / "data" / "raw"
PROCESSED = PROJECT_ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

EARLY_WINDOW_DAYS = 21  # weeks 1–3 (OULAD dates are days from course start)

print(f"Early window: day 0 – {EARLY_WINDOW_DAYS}")

## 1. Load raw tables & target

In [ ]:
student_info = pd.read_csv(RAW / "studentInfo.csv")
student_assessment = pd.read_csv(RAW / "studentAssessment.csv")
assessments = pd.read_csv(RAW / "assessments.csv")
student_vle = pd.read_csv(RAW / "studentVle.csv")
vle = pd.read_csv(RAW / "vle.csv")

student_info["at_risk"] = student_info["final_result"].isin(["Withdrawn", "Fail"]).astype(int)

keys = ["code_module", "code_presentation", "id_student"]
print(f"Enrollments: {len(student_info):,}  |  At-risk rate: {student_info['at_risk'].mean():.1%}")

## 2. Early VLE engagement features

In [ ]:
vle_early = student_vle[student_vle["date"] <= EARLY_WINDOW_DAYS].copy()
print(f"VLE rows in early window: {len(vle_early):,} / {len(student_vle):,}")

vle_early_agg = (
    vle_early.groupby(keys, as_index=False)
    .agg(
        early_clicks=("sum_click", "sum"),
        early_active_days=("date", "nunique"),
        early_first_activity=("date", "min"),
        early_last_activity=("date", "max"),
        early_unique_resources=("id_site", "nunique"),
    )
)

# Days since last login within the early window (staleness by end of week 3)
vle_early_agg["days_since_last_login"] = EARLY_WINDOW_DAYS - vle_early_agg["early_last_activity"]
vle_early_agg.loc[vle_early_agg["early_clicks"] == 0, "days_since_last_login"] = EARLY_WINDOW_DAYS

vle_early_agg.head()

In [ ]:
# Resource-type diversity (forum, content, url, etc.)
vle_labeled = vle_early.merge(
    vle[["id_site", "activity_type"]],
    on="id_site",
    how="left",
)

vle_diversity = (
    vle_labeled.groupby(keys, as_index=False)
    .agg(vle_diversity=("activity_type", "nunique"))
)

vle_early_agg = vle_early_agg.merge(vle_diversity, on=keys, how="left")
vle_early_agg["vle_diversity"] = vle_early_agg["vle_diversity"].fillna(0)
vle_early_agg.head()

## 3. Early assessment features

In [ ]:
sa = student_assessment.merge(assessments, on="id_assessment", how="inner")

# Only assessments due within the first 3 weeks
sa_early = sa[sa["date"] <= EARLY_WINDOW_DAYS].copy()
sa_early["late"] = np.where(
    sa_early["date"].notna(),
    sa_early["date_submitted"] > sa_early["date"],
    np.nan,
)

assess_early_agg = (
    sa_early.groupby(keys, as_index=False)
    .agg(
        early_mean_score=("score", "mean"),
        early_n_assessments=("score", "count"),
        early_late_count=("late", "sum"),
    )
)

# Submissions that happened early (even if due later) — activity signal
sa_submitted_early = sa[sa["date_submitted"] <= EARLY_WINDOW_DAYS]
submit_early_agg = (
    sa_submitted_early.groupby(keys, as_index=False)
    .agg(
        early_submissions=("score", "count"),
        early_submit_mean_score=("score", "mean"),
    )
)

assess_early_agg = assess_early_agg.merge(submit_early_agg, on=keys, how="left")
assess_early_agg.head()

## 4. Merge demographics + early features

In [ ]:
demo_cols = [
    "gender", "region", "highest_education", "imd_band",
    "age_band", "num_of_prev_attempts", "studied_credits", "disability",
]

df = (
    student_info[keys + ["final_result", "at_risk"] + demo_cols]
    .merge(vle_early_agg, on=keys, how="left")
    .merge(assess_early_agg, on=keys, how="left")
)

# Fill count features with 0 where student had no early activity
zero_fill = [
    "early_clicks", "early_active_days", "early_unique_resources",
    "vle_diversity", "early_n_assessments", "early_late_count",
    "early_submissions", "days_since_last_login",
]
df[zero_fill] = df[zero_fill].fillna(0)

# Students with no early VLE activity
df.loc[df["early_clicks"] == 0, ["early_first_activity", "early_last_activity"]] = np.nan

print(f"Feature table shape: {df.shape}")
print(f"At-risk rate: {df['at_risk'].mean():.1%}")
df.head()

## 5. Early-feature EDA

In [ ]:
early_numeric = [
    "early_clicks", "early_active_days", "vle_diversity",
    "early_mean_score", "early_submissions", "days_since_last_login", "at_risk",
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, title in zip(
    axes,
    ["early_clicks", "early_mean_score"],
    ["Early VLE clicks (weeks 1–3)", "Early mean assessment score"],
):
    plot_df = df.dropna(subset=[col]) if col == "early_mean_score" else df
    sns.histplot(
        data=plot_df, x=col, hue="at_risk", bins=30, kde=True, ax=ax,
        palette={0: "seagreen", 1: "crimson"}, common_norm=False,
    )
    ax.set_title(title)
plt.tight_layout()
plt.show()

In [ ]:
corr = df[early_numeric].corr()
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", center=0, ax=ax)
ax.set_title("Early-feature correlations")
plt.tight_layout()
plt.show()

print("Correlation with at_risk:")
print(corr["at_risk"].drop("at_risk").sort_values().round(3))

In [ ]:
summary_cols = [
    "early_clicks", "early_active_days", "vle_diversity",
    "early_mean_score", "early_submissions", "days_since_last_login",
]
df.groupby("at_risk")[summary_cols].agg(["mean", "median"]).round(2)

## 6. Save modeling-ready dataset

Feeds `03_model_training.ipynb`. Score columns may be NaN for students with no early submissions — tree models handle this; impute or drop for logistic regression.

In [ ]:
out_path = PROCESSED / "student_features_early.csv"
df.to_csv(out_path, index=False)

feature_cols = [c for c in df.columns if c not in keys + ["final_result", "at_risk"]]
print(f"Saved {df.shape[0]:,} rows → {out_path}")
print(f"Feature columns ({len(feature_cols)}): {feature_cols}")